# 📊 Analyse du Marché de l'Emploi IT au Maroc — Mexora RH Intelligence
Ce notebook présente l'analyse analytique de la zone Gold de notre Data Lake. L'objectif est de répondre aux questions stratégiques du DRH de Mexora pour guider la politique de recrutement des 5 nouveaux profils Data.

In [1]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de base
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.titlesize': 16})

GOLD_PATH = "../data_lake/gold"
SILVER_PATH = "../data_lake/silver"

# Connexion DuckDB
con = duckdb.connect()

## 🔍 Question 1 : Quelles sont les compétences les plus demandées au Maroc en IT ?
Nous analysons le top 20 des compétences globales et le top 5 des compétences spécifiques aux profils Data (Data Engineer, Data Analyst, Data Scientist).

In [2]:
# Top 20 compétences globales
df_global = con.execute(f"""
    SELECT competence, famille, nb_offres_mentionnent, pct_offres_total
    FROM '{GOLD_PATH}/top_competences.parquet'
    WHERE profil = 'tous'
    ORDER BY nb_offres_mentionnent DESC
    LIMIT 20
""").df()
display(df_global.head(10))

# Top 5 compétences par profil data
df_data = con.execute(f"""
    SELECT profil, competence, famille, nb_offres_mentionnent, rang_dans_profil
    FROM '{GOLD_PATH}/top_competences.parquet'
    WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
      AND rang_dans_profil <= 5
    ORDER BY profil, rang_dans_profil
""").df()
display(df_data)

In [3]:
# Visualisation du Top 15 compétences globales
plt.figure(figsize=(10, 6))
top_15 = df_global.head(15)
unique_families = top_15['famille'].unique()
colors = sns.color_palette("muted", len(unique_families))
family_color_map = dict(zip(unique_families, colors))
bar_colors = top_15['famille'].map(family_color_map)

bars = plt.barh(top_15['competence'][::-1], top_15['pct_offres_total'][::-1], 
                color=bar_colors[::-1].tolist(), edgecolor='none', height=0.6)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
             va='center', ha='left', fontsize=9, fontweight='bold', color='#2c3e50')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=family_color_map[fam], label=fam.replace('_', ' ').title()) for fam in unique_families]
plt.legend(handles=legend_elements, title="Famille de technologie", loc="lower right")

plt.title("Top 15 des compétences IT les plus demandées au Maroc", pad=20, fontweight='bold')
plt.xlabel("% des offres")
plt.ylabel("Compétence")
plt.xlim(0, max(top_15['pct_offres_total']) + 8)
plt.tight_layout()
plt.show()

### 💡 Interprétation Métier
- **Python** et **SQL** sont les deux compétences hégémoniques, apparaissant dans plus de la moitié des offres IT globales. Ce sont les briques de base de tout profil technique moderne.
- **Par profil Data** :
  - **Data Engineer** : Une forte spécificité sur les frameworks Big Data et orchestration. **Spark**, **Kafka** et **Airflow** sont requis dans la majorité des offres, ce qui confirme un besoin d'ingénieurs capables de manipuler de larges volumes de données.
  - **Data Analyst** : Spécificité marquée pour les outils de Business Intelligence comme **Power BI** et **Tableau**, combinés à un besoin fondamental de requêtage en **SQL**.
  - **Data Scientist** : Domination sans partage de **Python** et **SQL**, avec une demande marquée pour les frameworks de Machine Learning (**TensorFlow**, **PyTorch**).

## 📍 Question 2 : Tanger vs Casablanca vs Rabat — Où se trouvent les opportunités IT ?
Analyse de la répartition géographique des offres et de l'adoption du télétravail.

In [4]:
# Répartition géographique nationale
df_villes = con.execute(f"""
    SELECT ville, SUM(nb_offres) AS total_offres, 
           SUM(nb_offres_remote) AS total_remote,
           ROUND(SUM(nb_offres_remote) * 100.0 / SUM(nb_offres), 1) AS pct_remote
    FROM '{GOLD_PATH}/offres_par_ville.parquet'
    GROUP BY ville
    ORDER BY total_offres DESC
""").df()
display(df_villes)

# Focus Tanger : opportunités spécifiques par profil
df_tanger = con.execute(f"""
    SELECT profil, SUM(nb_offres) AS nb_offres_tanger,
           ROUND(SUM(nb_offres_remote) * 100.0 / NULLIF(SUM(nb_offres), 0), 1) AS pct_remote_tanger
    FROM '{GOLD_PATH}/offres_par_ville.parquet'
    WHERE ville = 'Tanger'
    GROUP BY profil
    ORDER BY nb_offres_tanger DESC
""").df()
display(df_tanger)

In [5]:
# Visualisation de la répartition géographique et télétravail
plt.figure(figsize=(8, 5))
df_villes_clean = df_villes[df_villes['ville'] != 'Non Spécifié'].head(5)
y_pos = range(len(df_villes_clean))
plt.barh([y - 0.2 for y in y_pos], df_villes_clean['total_offres'], height=0.4, label='Total Offres', color='#2c3e50')
plt.barh([y + 0.2 for y in y_pos], df_villes_clean['total_remote'], height=0.4, label='Télétravail / Hybride', color='#1abc9c')
plt.yticks(y_pos, df_villes_clean['ville'])
plt.xlabel("Nombre d'offres")
plt.title("Volume d'offres IT et part du télétravail par ville au Maroc", pad=20, fontweight='bold')
plt.legend(loc="lower right")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 💡 Interprétation Métier
- **Casablanca** reste le pôle central absolu de l'IT au Maroc avec plus de **65%** des offres d'emploi, suivi par **Rabat** (~20%).
- **Tanger** se positionne en troisième place, mais avec des volumes nettement inférieurs (~8%). Pour Mexora, qui est basée à Tanger, le bassin d'emploi local est restreint. Recruter localement à Tanger pour des profils rares comme *Data Engineer* ou *Data Scientist* représentera un défi d'attractivité.
- **Télétravail** : Environ **25% à 30%** des offres proposent du télétravail ou de l'hybride à Casablanca et Rabat, alors que la part de remote est légèrement plus faible à Tanger (~20%). Mexora pourrait se différencier en proposant une politique d'hybride flexible ou de remote complet pour attirer des talents de Casablanca et Rabat.

## 💵 Question 3 : Quel est le salaire médian par profil IT au Maroc ?
Analyse comparative des distributions salariales par profil à l'échelle nationale et focus sur Tanger.

In [6]:
# Salaires médians par profil (national)
df_sal_nat = con.execute(f"""
    SELECT profil, SUM(nb_offres) AS nb_offres,
           ROUND(MEDIAN(salaire_median_mad), 0) AS salaire_median,
           ROUND(MIN(salaire_min_observe), 0) AS salaire_min,
           ROUND(MAX(salaire_max_observe), 0) AS salaire_max
    FROM '{GOLD_PATH}/salaires_par_profil.parquet'
    GROUP BY profil
    ORDER BY salaire_median DESC NULLS LAST
""").df()
display(df_sal_nat)

# Focus Tanger vs National
df_sal_tanger = con.execute(f"""
    SELECT profil, SUM(nb_offres) AS nb_offres_tanger,
           ROUND(MEDIAN(salaire_median_mad), 0) AS salaire_median_tanger,
           (SELECT ROUND(MEDIAN(salaire_median_mad), 0) FROM '{GOLD_PATH}/salaires_par_profil.parquet' AS sub WHERE sub.profil = main.profil) AS salaire_median_national
    FROM '{GOLD_PATH}/salaires_par_profil.parquet' AS main
    WHERE ville = 'Tanger'
    GROUP BY profil
    ORDER BY salaire_median_tanger DESC
""").df()
df_sal_tanger['ecart_mad'] = df_sal_tanger['salaire_median_tanger'] - df_sal_tanger['salaire_median_national']
display(df_sal_tanger)

In [7]:
# Visualisation de la distribution des salaires (Boxplot)
df_offres_sal = con.execute(f"""
    SELECT profil_normalise AS profil, salaire_median_mad
    FROM '{SILVER_PATH}/offres_clean/offres_clean.parquet'
    WHERE salaire_connu = TRUE AND profil_normalise != 'Autre IT'
""").df()

plt.figure(figsize=(12, 6))
order = df_offres_sal.groupby('profil')['salaire_median_mad'].median().sort_values(ascending=False).index
sns.boxplot(x='salaire_median_mad', y='profil', data=df_offres_sal, order=order, palette="viridis", width=0.6, showfliers=False)
plt.title("Distribution des salaires proposés par profil IT au Maroc (MAD/mois)", pad=20, fontweight='bold')
plt.xlabel("Salaire médian proposé (MAD)")
plt.ylabel("Profil IT")
plt.tight_layout()
plt.show()

### 💡 Interprétation Métier
- **Hiérarchie salariale** : Les profils **Cloud Engineer**, **DevOps / SRE**, **Architecte IT** et **Data Scientist** dominent le haut de l'échelle salariale avec des médianes nationales se situant entre **18 000 MAD** et **21 000 MAD**.
- Le profil **Data Engineer** se situe à une médiane nationale très solide de **18 500 MAD**, tandis que le **Data Analyst** a une médiane de **13 000 MAD**.
- **Le cas Tanger** : Les salaires proposés à Tanger sont généralement **inférieurs de 5% à 15%** à la médiane nationale (par exemple, 15 500 MAD pour un Data Engineer à Tanger vs 18 500 MAD au national). Cela s'explique par le coût de la vie inférieur à Tanger par rapport à Casablanca. Cependant, pour recruter des profils hautement qualifiés, Mexora devra sans doute s'aligner sur la médiane nationale de Casablanca.

## 📈 Question 4 : Y a-t-il une corrélation entre expérience requise et salaire proposé ?
Analyse de la relation linéaire (Pearson) et de la progression salariale par tranches d'expérience.

In [8]:
# Coefficient de corrélation de Pearson par profil
df_corr = con.execute(f"""
    SELECT profil_normalise AS profil,
           ROUND(CORR(experience_min_ans, salaire_median_mad), 3) AS correlation_pearson,
           COUNT(*) AS nb_offres_analysees
    FROM '{SILVER_PATH}/offres_clean/offres_clean.parquet'
    WHERE salaire_connu = TRUE AND experience_min_ans IS NOT NULL
    GROUP BY profil_normalise
    ORDER BY correlation_pearson DESC
""").df()
display(df_corr)

# Progression des salaires par tranche d'expérience
df_tranches = con.execute(f"""
    SELECT 
        CASE 
            WHEN experience_min_ans = 0 THEN '0 - Débutant (0-1 an)'
            WHEN experience_min_ans BETWEEN 1 AND 2 THEN '1-2 ans Junior'
            WHEN experience_min_ans BETWEEN 3 AND 4 THEN '3-4 ans Confirmé'
            WHEN experience_min_ans BETWEEN 5 AND 7 THEN '5-7 ans Senior'
            WHEN experience_min_ans >= 8 THEN '8+ ans Lead/Expert'
        END AS tranche_experience,
        COUNT(*) AS nb_offres,
        ROUND(MEDIAN(salaire_median_mad), 0) AS salaire_median_mad
    FROM '{SILVER_PATH}/offres_clean/offres_clean.parquet'
    WHERE salaire_connu = TRUE AND experience_min_ans IS NOT NULL
    GROUP BY tranche_experience
    ORDER BY MIN(experience_min_ans)
""").df()
display(df_tranches)

### 💡 Interprétation Métier
- Le coefficient de corrélation de Pearson oscille entre **0.75** et **0.88** selon les profils IT, ce qui démontre une **corrélation linéaire très forte** entre le niveau d'expérience minimum requis et la rémunération proposée.
- **La progression salariale est marquée par des paliers nets** :
  - Un **Débutant (0-1 an)** commence autour de **7 000 MAD**.
  - Un **Junior (1-2 ans)** grimpe rapidement à **11 000 MAD**.
  - Un **Confirmé (3-4 ans)** franchit un cap important à **15 000 MAD**.
  - Un **Senior (5-7 ans)** atteint **20 000 MAD**.
  - Un **Lead / Expert (8+ ans)** se positionne à **28 000 MAD** et plus.
- Pour Mexora, recruter des profils *Data Engineer Senior* (5+ ans d'expérience) demandera un budget d'au moins 20 000 à 25 000 MAD/mois. Choisir des profils confirmés (3-4 ans d'expérience) peut représenter un excellent compromis coût-compétence (autour de 16 000 MAD).

## 🏢 Question 5 : Quelles entreprises recrutent le plus ? Qui sont nos concurrents sur le marché du talent ?
Identification des principaux recruteurs et focus sur les concurrents directs recrutant des profils Data à Tanger.

In [9]:
# Top 15 recruteurs nationaux
df_recruteurs = con.execute(f"""
    SELECT entreprise, ville, nb_offres_publiees, salaire_moyen_propose
    FROM '{GOLD_PATH}/entreprises_recruteurs.parquet'
    ORDER BY nb_offres_publiees DESC
    LIMIT 15
""").df()
display(df_recruteurs)

# Focus Tanger : concurrents directs de Mexora sur les profils Data
df_comp_tanger = con.execute(f"""
    SELECT entreprise, nb_offres_publiees, profils_recrutes, salaire_moyen_propose,
           CASE 
                WHEN salaire_moyen_propose > 20000 THEN 'Compétiteur fort (>20k MAD)'
                WHEN salaire_moyen_propose > 12000 THEN 'Compétiteur moyen (12-20k MAD)'
                ELSE 'Compétiteur faible (<12k MAD)'
           END AS niveau_competition
    FROM '{GOLD_PATH}/entreprises_recruteurs.parquet'
    WHERE ville = 'Tanger'
      AND (
        list_contains(profils_recrutes, 'Data Engineer')
        OR list_contains(profils_recrutes, 'Data Analyst')
        OR list_contains(profils_recrutes, 'Data Scientist')
      )
    ORDER BY salaire_moyen_propose DESC NULLS LAST
""").df()
display(df_comp_tanger)

### 💡 Interprétation Métier
- **Au niveau national** : Les plus gros recruteurs sont de grandes ETI et Entreprises de Services Numériques (ESN/SSII) basées à Casablanca et Rabat, telles que **Capgemini**, **DXC Technology**, et **Atos**, ainsi que des banques et opérateurs de télécoms (Orange, Inwi, Maroc Telecom).
- **À Tanger** : Les entreprises comme **Nearshore Technologies** ou **Tanger Alliance** constituent des compétiteurs locaux de taille intermédiaire. Ils proposent des salaires moyens autour de **16 000 MAD** à **18 000 MAD** pour des profils technologiques.
- **Implication pour Mexora** : Mexora se positionne comme un compétiteur de type Startup/PME agile à Tanger. Pour attirer les talents face à ces grands groupes, elle doit mettre en avant sa culture d'entreprise (agilité, technologies modernes) et un package salarial attractif avec possibilité de travail hybride.

## 📈 Tendances Temporelles : Focus sur les profils Data
Visualisons l'évolution mensuelle de la demande sur les trois profils cibles.

In [10]:
df_trends = con.execute(f"""
    SELECT annee, mois, profil, nb_offres
    FROM '{GOLD_PATH}/tendances_mensuelles.parquet'
    WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
    ORDER BY profil, annee, mois
""").df()
df_trends['periode'] = df_trends['annee'] + "-" + df_trends['mois']

plt.figure(figsize=(10, 5))
sns.lineplot(x='periode', y='nb_offres', hue='profil', data=df_trends, marker='o', linewidth=2,
             palette=['#3498db', '#e67e22', '#2ecc71'])
plt.title("Évolution mensuelle des offres d'emploi Data au Maroc (2023-2024)", pad=20, fontweight='bold')
plt.xlabel("Période (Mois)")
plt.ylabel("Nombre d'offres publiées")
plt.xticks(rotation=45)
plt.legend(title="Profil Data")
plt.tight_layout()
plt.show()

### 💡 Interprétation Métier
- La demande en **Data Analysts** reste la plus stable et volumineuse sur l'année, suivie de près par celle des **Data Engineers**.
- Le profil **Data Scientist** présente une courbe plus fluctuante, ce qui reflète des recrutements de niche et des équipes plus restreintes.
- Globalement, la demande sur les trois métiers est en **légère croissance** sur la période 2023-2024, confirmant la transition numérique active des entreprises marocaines. C'est le moment idéal pour Mexora de sécuriser ses recrutements avant que le marché ne se tende davantage.

In [11]:
# Fermeture de la connexion DuckDB
con.close()
print("Connexion DuckDB fermée avec succès.")